# Трансформеры для NLP

## install and imports

In [ ]:
!pip -q install "transformers>=4.44.0" "datasets>=2.19.0" "accelerate>=0.34.0" "evaluate>=0.4.2"   "peft>=0.12.0" "bitsandbytes>=0.43.0" --upgrade
import torch, transformers, datasets, evaluate, peft
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
device


## pipeline

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis",
                     model="distilbert-base-uncased-finetuned-sst-2-english")
sentiment(["I love cats!", "This is disgasting"])

In [ ]:
mlm = pipeline("fill-mask", model="bert-base-uncased")
mlm("It is the [MASK] country in the world.")

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="facebook/bart-large-mnli")
zero_shot("I need a refund for my last order — it arrived broken.",
          candidate_labels=["refund", "shipping", "product quality", "other"])


In [ ]:
summ = pipeline("summarization", model="google/flan-t5-small")
text = '''
The university's birthday is considered to be 26 March 1900, when a Mechanics, Optics and Watchmaking Department was opened in the Prince Nicholas Vocational School. At the time it was the only school in the Russian Empire that prepared specialists in these areas. The first year, some 65 applications were received for 30 places. Eighteen students were admitted to Watchmaking and 18 to Mechanics and Optics sections
'''
summ(text, max_new_tokens=60)

In [ ]:
translator = pipeline("translation_en_to_ru", model="Helsinki-NLP/opus-mt-en-ru")
translator("You can use it.")

## BERT


In [ ]:
from datasets import load_dataset

ds = load_dataset("glue", "sst2")
ds

In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize(batch):
    return tok(batch["sentence"], truncation=True, padding="max_length", max_length=128)

In [ ]:
tok_ds = ds.map(tokenize, batched=True)
tok_ds = tok_ds.rename_column("label", "labels")
tok_ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
tok_ds

In [ ]:
import evaluate, numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
args = TrainingArguments(
    output_dir="bert-sst2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,   # для демо 1 эпохи достаточно
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds["train"].shuffle(seed=42).select(range(5000)),  # укороченный train для скорости
    eval_dataset=tok_ds["validation"],
    tokenizer=tok,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
samples = tok_ds["validation"].select(range(5))
preds = trainer.predict(samples)

import numpy as np
labels = np.argmax(preds.predictions, axis=-1)

for text, true_label, pred_label in zip(ds["validation"]["sentence"][:5], preds.label_ids[:5], labels[:5]):
    print(f"Текст: {text}")
    print(f"Реальная метка: {true_label}, Предсказанная: {pred_label}")
    print("-"*60)


## GPT


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

gpt_model = "gpt2"
gen = pipeline("text-generation", model=gpt_model, device=0 if torch.cuda.is_available() else -1)

prompt = "The university's birthday is considered to be 26 March 1900, when a Mechanics, Optics and Watchmaking Department was opened in "
gen(prompt, max_new_tokens=60, do_sample=True, temperature=0.9, top_p=0.9)

- **`max_new_tokens`** — максимальное число новых токенов, которое модель сгенерирует.
- **`temperature`** (0–2) — «дерзость» сэмплирования. Ниже — консервативнее, выше — более разнообразно.
T < 1 → модель более уверена и консервативна, чаще выбирает самые вероятные токены.
T = 1 → без изменений.
T > 1 → модель становится более креативной, но иногда теряет связность.
- **`top_k`** — сэмплируем только из `k` самых вероятных токенов. На каждом шаге сортируем вероятности токенов по убыванию и берём только k самых вероятных.
- **`top_p`** — nucleus sampling: сэмплируем из наименьшего множества токенов, чья суммарная вероятность ≥ `p`.
- **`repetition_penalty`** — штраф за повторы; >1.0 уменьшает зацикливание.
- **`do_sample`** — если `False`, используется «жадная»/beam генерация; если `True`, включается стохастическое сэмплирование.
- **`num_beams`** — количество лучей для beam search;


## LLM


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
try:
    tok_chat = AutoTokenizer.from_pretrained(model_id)
    chat_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
    use_tinyllama = True
except Exception as e:
    print("Fallback to gpt2 due to:", e)
    tok_chat = AutoTokenizer.from_pretrained("gpt2")
    chat_model = AutoModelForCausalLM.from_pretrained("gpt2")
    use_tinyllama = False

In [ ]:
def chat(messages, **genkw):
    if hasattr(tok_chat, "apply_chat_template"):
        prompt = tok_chat.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = ""
        for m in messages:
            prompt += f"{m['role'].upper()}: {m['content']}"
        prompt += "ASSISTANT: "
    inputs = tok_chat(prompt, return_tensors="pt").to(chat_model.device)
    out = chat_model.generate(**inputs, max_new_tokens=genkw.get("max_new_tokens", 128),
                              do_sample=genkw.get("do_sample", True),
                              temperature=genkw.get("temperature", 0.7),
                              top_p=genkw.get("top_p", 0.9),
                              top_k=genkw.get("top_k", 50),
                              repetition_penalty=genkw.get("repetition_penalty", 1.1))
    text = tok_chat.decode(out[0], skip_special_tokens=True)
    print(text)

chat([
    {"role":"system","content":"You are a concise assistant."},
    {"role":"user","content":"Объясни разницу между BERT и GPT."}
], max_new_tokens=256, temperature=0.7, top_p=0.9)


- `stop_sequences` / специальные токены — позволяют аккуратно обрезать вывод.
- `presence_penalty`, `frequency_penalty` — аналоги.


Extra:
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding - https://arxiv.org/abs/1810.04805
- Improving Language Understanding by Generative Pre-Training - https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf
- Language Models are Unsupervised Multitask Learners - https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
- Attention Is All You Need - https://arxiv.org/abs/1706.03762